In [1]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Model structure
class Net(nn.Module):
    def __init__(self):
        # 在构造函数中，实例化不同的layer组件，并赋给类成员变量
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # 在前馈函数中，利用实例化的组件对网络进行搭建，并对输入Tensor进行操作，并返回Tensor类型的输出结果
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5) # n * 400 也可以用函数flatten(1,dim=?)
        x = F.relu(self.fc1(x)) 
        x = F.relu(self.fc2(x))
        x = self.fc3(x) 
        return x


In [3]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Running Device:', device)

Running Device: cpu


In [ ]:
net = Net().to(device) #把模型搬运到GPU上去
print(net) #这里是在打印nn.Module给类配备的魔法函数

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [ ]:
# 对获取的图像数据做ToTensor()变换和归一化
# ToTensor能够直接把变量转化为0-1之间的变量吗？可以的，真是神奇！
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# 利用torchvision提供的CIFAR10数据集类，实例化训练集和测试集提取类
# 后边的train = 实际上是在规定这个变量是测试集还是训练集，因为CIFAR10是一个单独设立的包
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)

# 利用torch提供的DataLoader, 实例化训练集DataLoader 和 测试集DataLoader
Batch_size = 2 # 可以改变，改的是每一次进入训练过程中的图片数量
trainLoader = torch.utils.data.DataLoader(trainset, batch_size=Batch_size, shuffle=True, num_workers=2)
testLoader = torch.utils.data.DataLoader(testset, batch_size=Batch_size, shuffle=False, num_workers=2)

# CIFAR10 类别内容
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print("OK")

OK


In [ ]:
#测试 trainLoader，读取一个batch的数据
# 这里是在示范trainLoader的储存形式和表达形式
# trainLoader里储存的是一个个tensor

data_iter = iter(trainLoader)
inputs, labels = next(data_iter)

print("inputs.shape = ", inputs.shape)
print("labels.shape = ", labels.shape)
print("inputs = ", inputs)
print("labels = ", labels)


inputs.shape =  torch.Size([2, 3, 32, 32])
labels.shape =  torch.Size([2])
inputs =  tensor([[[[ 0.3490,  0.3569,  0.3647,  ..., -0.0431, -0.0510, -0.0353],
          [ 0.4431,  0.5216,  0.5059,  ..., -0.0510, -0.0431, -0.0431],
          [ 0.2706,  0.3490,  0.3176,  ..., -0.0431, -0.0353, -0.0275],
          ...,
          [-0.2706,  0.7804,  1.0000,  ...,  0.9922,  0.9843,  1.0000],
          [ 0.3569,  0.9137,  0.9843,  ...,  0.9922,  0.9922,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000]],

         [[ 0.2863,  0.2863,  0.2784,  ..., -0.0588, -0.0588, -0.0510],
          [ 0.3647,  0.4353,  0.4118,  ..., -0.0824, -0.0745, -0.0667],
          [ 0.2078,  0.2706,  0.2471,  ..., -0.0824, -0.0745, -0.0667],
          ...,
          [-0.1765,  0.8039,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 0.4196,  0.9216,  0.9922,  ...,  0.9843,  0.9843,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000]],

         [[ 0.364

In [13]:
#一个批次做前向，并且预测标签
outputs = net(inputs) #随机初始化的
print(outputs) #2 * 10 因为输入是两张照片

tensor([[-0.0153,  0.0080, -0.0824, -0.0919, -0.0641, -0.0311,  0.0900, -0.0352,
         -0.1286, -0.0739],
        [-0.0183,  0.0115, -0.0777, -0.0925, -0.0641, -0.0276,  0.0874, -0.0235,
         -0.1267, -0.0713]], grad_fn=<AddmmBackward0>)


In [ ]:
#一个批次做前向，并且预测标签；如果是测试阶段，则不需要保留导数、创建计算图
with torch.no_grad(): #这里没有保留梯度，这是一种python自带的保留形式，相当于是说这一块里这一变量按照这个形式输出
    outputs = net(inputs)
    print(outputs)

tensor([[-0.0153,  0.0080, -0.0824, -0.0919, -0.0641, -0.0311,  0.0900, -0.0352,
         -0.1286, -0.0739],
        [-0.0183,  0.0115, -0.0777, -0.0925, -0.0641, -0.0276,  0.0874, -0.0235,
         -0.1267, -0.0713]])


In [15]:
#获得一个批次的预测
_, predicted = torch.max(outputs.data, 1) # max()会返回两个量，第一个是数字本身，第二个是数字的index
print(predicted)

predicted = torch.argmax(outputs.data, 1)# 1 代表从哪个维度去取最大
print(predicted)

tensor([6, 6])
tensor([6, 6])


In [ ]:
#计算一个批次的CrossEntropyLoss 首先声明这个函数
criterion = nn.CrossEntropyLoss()

print(outputs)
print(labels)

loss = criterion(outputs, labels)#前面是需要对比的没有处理的特征数据（还没有变成分布），后面是one-hot标签输入

print(loss.item())

tensor([[-0.0153,  0.0080, -0.0824, -0.0919, -0.0641, -0.0311,  0.0900, -0.0352,
         -0.1286, -0.0739],
        [-0.0183,  0.0115, -0.0777, -0.0925, -0.0641, -0.0276,  0.0874, -0.0235,
         -0.1267, -0.0713]])
tensor([0, 3])
2.3167989253997803


In [ ]:
#计算准确率

print(predicted == labels)
print((predicted == labels).shape)
print((predicted == labels).squeeze().shape)
correct = (predicted == labels).sum().item()#原来还能这样
acc = correct / labels.size()[0]
print(acc)



tensor([False, False])
torch.Size([2])
torch.Size([2])
0.0


In [ ]:
#遍历 trainLoader，逐一读取 trainLoader能够直接遍历，enumerate则是给每一个输出编号
for i, data in enumerate(trainLoader, 0):
    if i > 2: #这里为了演示，分批次打印出了每一个trainLoader里的内容
        break 
    inputs, labels = data
    print("i = ", i )
    print("inputs.shape = ", inputs.shape)
    print("labels.shape = ", labels.shape)
    print("inputs = ", inputs)
    print("labels = ", labels)
    
    # 将数据迁移到device中，如device为GPU，则将数据从CPU迁移到GPU；如device为CPU，则将数据从CPU迁移到CPU（即不作移动）
    inputs, labels = inputs.to(device), labels.to(device) 

i =  0
inputs.shape =  torch.Size([2, 3, 32, 32])
labels.shape =  torch.Size([2])
inputs =  tensor([[[[ 0.6078,  0.6235,  0.6471,  ...,  0.8039,  0.8039,  0.8039],
          [ 0.6549,  0.6549,  0.6784,  ...,  0.8431,  0.8431,  0.8510],
          [ 0.6627,  0.6549,  0.6706,  ...,  0.8431,  0.8510,  0.8588],
          ...,
          [ 0.0510,  0.0510,  0.0588,  ...,  0.0431,  0.0353,  0.0275],
          [ 0.0431,  0.0510,  0.0510,  ...,  0.0667,  0.0510,  0.0353],
          [ 0.0275,  0.0196,  0.0275,  ...,  0.0588,  0.0510,  0.0275]],

         [[ 0.7569,  0.7725,  0.7961,  ...,  0.8745,  0.8745,  0.8667],
          [ 0.8039,  0.8039,  0.8275,  ...,  0.8980,  0.8902,  0.8902],
          [ 0.7961,  0.7961,  0.8039,  ...,  0.8745,  0.8745,  0.8824],
          ...,
          [ 0.0275,  0.0275,  0.0275,  ..., -0.0745, -0.0510, -0.0353],
          [-0.0039,  0.0039,  0.0039,  ..., -0.0353, -0.0275, -0.0275],
          [-0.0353, -0.0353, -0.0275,  ..., -0.0196, -0.0196, -0.0275]],

         [

In [14]:
#遍历 testLoader，逐一读取
for i, data in enumerate(testLoader, 0):
    if i > 0:
        break 
    inputs, labels = data
    print("inputs.shape = ", inputs.shape)
    print("labels.shape = ", labels.shape)
    print("inputs = ", inputs)
    print("labels = ", labels)
    
    # 将数据迁移到device中，如device为GPU，则将数据从CPU迁移到GPU；如device为CPU，则将数据从CPU迁移到CPU（即不作移动）
    inputs, labels = inputs.to(device), labels.to(device)

inputs.shape =  torch.Size([2, 3, 32, 32])
labels.shape =  torch.Size([2])
inputs =  tensor([[[[ 0.2392,  0.2471,  0.2941,  ...,  0.0745, -0.0118, -0.0902],
          [ 0.1922,  0.1843,  0.2471,  ...,  0.0667, -0.0196, -0.0667],
          [ 0.1843,  0.1843,  0.2392,  ...,  0.0902,  0.0196, -0.0588],
          ...,
          [-0.4667, -0.6706, -0.7569,  ..., -0.7020, -0.8980, -0.6863],
          [-0.5216, -0.6157, -0.7255,  ..., -0.7961, -0.7725, -0.8431],
          [-0.5765, -0.5608, -0.6471,  ..., -0.8118, -0.7333, -0.8353]],

         [[-0.1216, -0.1294, -0.0902,  ..., -0.2549, -0.2863, -0.3333],
          [-0.1216, -0.1373, -0.1059,  ..., -0.2549, -0.2863, -0.3098],
          [-0.1373, -0.1451, -0.1294,  ..., -0.2314, -0.2549, -0.3020],
          ...,
          [-0.0275, -0.2157, -0.3098,  ..., -0.2392, -0.4980, -0.3333],
          [-0.0902, -0.2000, -0.3333,  ..., -0.3569, -0.3569, -0.4980],
          [-0.1608, -0.1765, -0.3020,  ..., -0.3961, -0.3412, -0.4745]],

         [[-0.615